# Task 1: LLM-Powered Equity Research Assistant

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gobishangar1/CDAZZDEV-MLE-GobishangarVijayendran/blob/main/task1_financial/notebooks/task1_equity_research.ipynb)

**Candidate:** Gobishangar Vijayendran · **Ticker:** NVDA (liquid, with plenty of news)

This notebook runs every graded part of Task 1, in rubric order. All logic lives in the
`equity_research` package; the notebook only calls it and shows the results.

| Part | Section of this notebook |
|---|---|
| 1A OHLCV fetch, indicators, news, summary, robustness | Sections 1-6 |
| 1B per-headline JSON, signal reasoning, validation, prompts | Sections 7-12 |
| Bonus report | Section 13 |

Design: a fixed LangGraph workflow (not an agent). **Python does the maths; the LLM only judges.**

## 0. Setup

In [ ]:
import os, sys

if "google.colab" in sys.modules:                       # Colab: clone the repo and install
    if not os.path.exists("CDAZZDEV-MLE-GobishangarVijayendran"):
        !git clone -q https://github.com/Gobishangar1/CDAZZDEV-MLE-GobishangarVijayendran.git
    %cd CDAZZDEV-MLE-GobishangarVijayendran/task1_financial
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":       # local Jupyter started inside notebooks/
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
# The Groq key comes from Colab Secrets (key icon on the left) or from the environment.
# It is NEVER printed and NEVER written to a file.
from equity_research.config import load_colab_secret, require_groq_key, setup_logging

loaded = load_colab_secret("GROQ_API_KEY")
require_groq_key()
print("GROQ_API_KEY available:", True, "| loaded from Colab Secrets:", loaded)

setup_logging()        # our logs at INFO; validation failures appear as WARNING lines
TICKER = "NVDA"

## 1. Configuration: no magic numbers
Every window, threshold and weight is a named constant in `config.py`, each with a one-line reason.

In [ ]:
import pandas as pd
from equity_research.config import constants_table
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(constants_table(), columns=["constant", "value"])

## 2. Task 1A: OHLCV data (at least two years, no hard-coded dates)
Start date = today − 2 years − 300 days of warm-up (so SMA-200 has a value on day 1 of the 2-year window).
Prices are split- and dividend-adjusted.

In [ ]:
from equity_research.market_data import compute_start_dates, fetch_prices

df_raw, price_warnings = fetch_prices(TICKER)
fetch_start, display_start = compute_start_dates(pd.Timestamp.today())
in_window = df_raw[df_raw.index >= display_start]

print(f"Fetched from {df_raw.index[0].date()} to {df_raw.index[-1].date()}: {len(df_raw)} trading days")
print(f"2-year window from {display_start.date()}: {len(in_window)} trading days "
      f"({(in_window.index[-1] - in_window.index[0]).days / 365.25:.2f} years)")
print("Data-quality warnings:", price_warnings or "none")
assert (in_window.index[-1] - in_window.index[0]).days >= 365 * 2 - 7
df_raw.tail()

## 3. Task 1A: technical indicators from first principles (no TA-Lib)
SMA-50, SMA-200, RSI-14 (Wilder), MACD (12, 26, 9), Bollinger Bands (20, 2σ, population std).

In [ ]:
from equity_research.indicators import add_indicators

df = add_indicators(df_raw)
cols = ["Close", "SMA_50", "SMA_200", "RSI_14", "MACD", "MACD_signal", "MACD_hist",
        "BB_lower", "BB_middle", "BB_upper", "BB_pctB"]
df[cols].tail().round(3)

**Proof of accuracy:** the test suite checks hand-computed values (SMA, EMA, Wilder RSI), an independent
loop implementation of EMA, MACD identities, Bollinger against numpy's population std, and edge cases
(flat prices, only gains, too little data). The full suite runs offline in a few seconds.

In [ ]:
!python -m pytest -q tests

## 4. Task 1A: chart preview (last 12 months)

In [ ]:
from pathlib import Path
from IPython.display import Image, display
from equity_research.report import make_chart

Path("outputs").mkdir(exist_ok=True)
display(Image(filename=str(make_chart(df, TICKER, Path("outputs") / "preview_chart.png"))))

## 5. Task 1A: news headlines (at least ten, free sources)
Provider chain: yfinance news → Yahoo Finance RSS → Google News RSS, de-duplicated, newest first.
Free feeds mix in general market stories, so headlines that **name the ticker or the company** come first;
general headlines are only added to reach ten, and that is reported as a warning.

In [ ]:
from equity_research.market_data import fetch_info
from equity_research.news import fetch_news

headlines, news_warnings = fetch_news(TICKER, fetch_info(TICKER).get("longName"))
print(f"{len(headlines)} unique headlines, {sum(h['relevant'] for h in headlines)} about {TICKER} "
      f"| warnings: {news_warnings or 'none'}")
pd.DataFrame(headlines)[["published_at", "publisher", "source", "relevant", "title"]]

## 6. Task 1A: the summary dictionary
Current price, 52-week high/low (from the High/Low columns), P/E (Yahoo trailing → price/EPS if EPS > 0 →
None), YTD return (vs the last close of the previous year), and a rule-based momentum signal.

In [ ]:
from pprint import pprint
from equity_research.analysis import build_summary, indicator_snapshot, momentum_signal
from equity_research.market_data import fetch_info

info = fetch_info(TICKER)
snapshot = indicator_snapshot(df)
summary = build_summary(TICKER, df, info, momentum_signal(snapshot)).model_dump()
pprint(summary, sort_dicts=False)

## 7. Task 1B: the LangGraph workflow
Parallel fetches → `Send()` fan-out (one LLM call per headline) → joins → signal with a validation retry
loop → rule-based fallback → report. Nodes write state; routers only read it.

In [ ]:
from equity_research.graph import build_graph, run_pipeline

graph = build_graph(None, None)
print(graph.get_graph().draw_mermaid())
try:                                    # the PNG uses an online renderer; the text above always works
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("PNG render skipped:", type(exc).__name__)

### Run the whole pipeline
Each line shows a node finishing, with the elapsed time. Headline calls are rate-limited for the Groq free tier.

In [ ]:
result = run_pipeline(TICKER, output_dir="outputs")
print("\nstatus:", result["status"], "| signal source:", result["signal_source"])

### Trace of this run (`trace.jsonl`)
One event per node and per LLM call: which model answered (20B, or the 120B fallback), latency, tokens,
and every validation result. Saved next to `run.json`.

In [ ]:
import json
print(json.dumps(result["trace_summary"], indent=2))
events = [json.loads(line) for line in open(result["report_paths"]["trace"], encoding="utf-8")]
llm_calls = pd.DataFrame([e for e in events if e["kind"] == "llm_call"])
llm_calls[["t_s", "purpose", "attempt", "model", "latency_ms", "input_tokens", "output_tokens", "ok", "detail"]]

## 8. Task 1B: per-headline structured JSON + aggregate score
Each headline → one LLM call → JSON validated by Pydantic (`headline`, `sentiment`, `confidence`,
`brief_reason`). Invalid replies are logged with the raw text and retried once with the error message.

Aggregate: **S = Σ wᵢ·cᵢ·sᵢ / Σ wᵢ·cᵢ**, with s = +1 / 0 / −1, c = confidence,
w = 0.5^(age_days / 3) (3-day half-life). Neutral headlines stay in the denominator, so mostly-neutral
news gives a score near 0.

In [ ]:
import json
per_headline = pd.DataFrame(result["headline_results"])
print("Example of one validated JSON object:")
print(json.dumps({k: result["headline_results"][0][k] for k in ["headline", "sentiment", "confidence", "brief_reason"]},
                 indent=2))
print("\nFailures (logged, run continued):", result["headline_failures"] or "none")
print("\nAggregate:", json.dumps(result["sentiment"], indent=2))
per_headline[["headline", "sentiment", "confidence", "brief_reason", "publisher", "published_at"]]

## 9. Task 1B: facts and conflicts given to the LLM
Code turns numbers into **facts** (direction, distance, zone, change over time) and lists **conflicts**
(signals that disagree). The prompt requires the LLM to resolve every conflict, so it must reason about
combinations instead of echoing values, and it never has to do arithmetic.

In [ ]:
print("FACTS:")
for f in result["facts"]:
    print(" -", f)
print("\nCONFLICTS:")
for c in result["conflicts"] or ["none"]:
    print(" -", c)

## 10. Task 1B: the Buy / Hold / Sell signal and its validation
Two levels: **schema** (Pydantic: enum, 0-1 confidence, 3-5 sentences, ≥ 2 drivers) and **meaning**
(no invented numbers, ≥ 2 indicator groups, conflicts addressed, confidence ≤ 0.75 when conflicts exist,
anti-echo). Up to 3 attempts with feedback, then a rule-based fallback.

In [ ]:
print(json.dumps(result["signal"], indent=2))
print("\nSource:", result["signal_source"])
pd.DataFrame([{"attempt": h["attempt"], "passed": not h["problems"], "problems": "; ".join(h["problems"]) or "-"}
              for h in result["signal_history"]])

## 11. Fault injection: invalid LLM output is caught, logged and handled
Fake LLMs (no API calls) return broken output on purpose:
1. a headline reply that is not JSON, then a valid one → **repair retry works**;
2. an "echo" signal → **the validator lists every problem**;
3. a signal LLM that is always broken → **3 logged attempts, then the rule-based fallback**.

In [ ]:
from langchain_core.messages import AIMessage
from equity_research.schemas import TradeSignal
from equity_research.sentiment import classify_headline
from equity_research.trade_signal import validate_signal

class FakeLLM:
    def __init__(self, replies): self.replies, self.calls = list(replies), 0
    def invoke(self, prompt):
        self.calls += 1
        return AIMessage(content=self.replies[min(self.calls, len(self.replies)) - 1])

item = headlines[0]
valid = json.dumps({"headline": item["title"], "sentiment": "neutral", "confidence": 0.6, "brief_reason": "demo"})
res, fail = classify_headline(FakeLLM(['Sure! {"sentiment": "bullish", "confidence": 1.4}', valid]), TICKER, None, item)
print("1) repaired result:", res.model_dump() if res else fail, "\n")

echo = TradeSignal(signal="BUY", confidence=0.95, key_drivers=["RSI", "MACD"], conflicting_evidence=[],
                   justification="The RSI is 99.9. The MACD is 12345. The price is up.")
print("2) validator problems for an echo answer:")
for p in validate_signal(echo, result["facts"], result["conflicts"] or ["demo conflict"]):
    print("   -", p)

In [ ]:
# 3) Full graph with a signal LLM that never returns JSON (headline LLM is also fake: no API cost)
fake_headlines = FakeLLM(['{"headline": "x", "sentiment": "neutral", "confidence": 0.5, "brief_reason": "demo"}'])
broken_signal = FakeLLM(["I think you should buy."])
demo = run_pipeline(TICKER, fake_headlines, broken_signal, output_dir=None)
print("\nsignal source:", demo["signal_source"], "| attempts:", len(demo["signal_history"]))
print(json.dumps(demo["signal"], indent=2))

## 12. Robustness: bad ticker and missing data
An unknown ticker stops **before any LLM call** with a clean error report (no traceback). Missing or
broken Yahoo fields become `None` with a source label.

In [ ]:
from equity_research.analysis import pe_ratio
bad = run_pipeline("XXXXZZ", FakeLLM(["{}"]), FakeLLM(["{}"]), output_dir=None)
print(json.dumps(bad, indent=2))
print("\nP/E with Yahoo returning 'Infinity' and negative EPS:", pe_ratio(150.0, {"trailingPE": "Infinity", "trailingEps": -2.0}))

## 13. Bonus: one-page equity research brief
Company snapshot, technical outlook (embedded matplotlib chart), news sentiment with the top three headlines,
the recommendation with its reasoning, data-quality notes, and a risk disclaimer.
Files: `brief.md`, `brief.html` (self-contained), `chart.png`, `run.json`.

In [ ]:
from IPython.display import HTML
print(json.dumps(result["report_paths"], indent=2))
HTML(open(result["report_paths"]["html"], encoding="utf-8").read())

In [ ]:
# Colab only: download the outputs folder, then commit it to task1_financial/outputs/
if "google.colab" in sys.modules:
    from google.colab import files
    !zip -q -r outputs.zip outputs
    files.download("outputs.zip")

## Appendix: every prompt (defined as constants in `prompts.py`)

In [ ]:
from equity_research.prompts import all_prompts
for name, text in all_prompts().items():
    print(f"===== {name} =====\n{text}\n")